# Bonsai Image — Colab + Gradio Share (minimal)\nSingle cell. Prints the Gradio share URL when ready.

In [ ]:
# setup + launch\nimport os\nimport subprocess\nimport json\nimport time\nfrom pathlib import Path\n\nROOT = Path('.').resolve()\nREPO = ROOT / 'Bonsai-Image-Demo'\nVENV_PY = REPO / '.venv' / 'bin' / 'python'\nLAUNCHER = REPO / 'bonsai_gradio_launcher.py'\nURL_FILE = Path('/tmp/bonsai_gradio_url.json')\n\ndef run(cmd):\n    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)\n    if p.stdout:\n        print(p.stdout.rstrip())\n    if p.returncode != 0:\n        print(f'FAILED ({p.returncode}): {cmd}')\n        if p.stderr:\n            print(p.stderr.rstrip())\n    return p.returncode\n\n# 1) clone\nif not REPO.exists():\n    run('git clone https://github.com/PrismML-Eng/Bonsai-Image-Demo.git')\n\n# 2) setup\nrun(f'chmod +x {REPO}/setup.sh')\nrun(f'cd {REPO} && ./setup.sh')\n\n# 3) dry-run import check\ndry = subprocess.run(\n    [str(VENV_PY), '-c', 'import backend_gpu.server; print(\"backend_gpu ok\")'],\n    capture_output=True,\n    text=True,\n    cwd=str(REPO),\n)\nprint('VENV_PY:', VENV_PY)\nprint('dry-run:', dry.stdout.strip(), dry.stderr.strip(), 'rc=', dry.returncode)\nif dry.returncode != 0:\n    raise SystemExit(1)\n\n# 4) write launcher (non-blocking launch + polling)\nLAUNCHER.write_text(f'''\nimport os, sys, json, time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path('.').resolve()))\n\nfrom backend_gpu.server import build_pipeline\npipe = build_pipeline(model_id='prism-ml/bonsai-image-ternary-4B-gemlite-2bit')\n\nimport gradio as gr\nfrom PIL import Image\n\ndef generate(prompt, seed, steps, width, height):\n    seed = int(seed) if seed not in (None, '', 'None') else None\n    steps = int(steps)\n    width = int(width)\n    height = int(height)\n    out = pipe(prompt=prompt, num_inference_steps=steps, guidance_scale=1.0, height=height, width=width)\n    img = out.images[0] if hasattr(out, 'images') else out\n    return img, seed\n\nwith gr.Blocks(title='Bonsai Image') as demo:\n    gr.Markdown('# Bonsai Image Generation API')\n    with gr.Row():\n        with gr.Column():\n            prompt = gr.Textbox(label='Prompt', placeholder='A bonsai tree in a quiet ceramic studio, soft morning light')\n            seed = gr.Number(label='Seed', value=None, precision=0)\n            steps = gr.Slider(1, 20, value=4, step=1, label='Steps')\n            width = gr.Slider(256, 1024, value=512, step=32, label='Width')\n            height = gr.Slider(256, 1024, value=512, step=32, label='Height')\n            btn = gr.Button('Generate')\n        with gr.Column():\n            output = gr.Image(type='pil', label='Generated Image')\n            seed_out = gr.Number(label='Seed Used', interactive=False)\n    btn.click(generate, inputs=[prompt, seed, steps, width, height], outputs=[output, seed_out])\n\ndemo.queue()\nlaunched = demo.launch(share=True, quiet=True, prevent_thread_lock=True, server_name='0.0.0.0', server_port=7860)\n\n# Poll share_url\nshare_url = launched\nfor _ in range(120):\n    share_url = getattr(demo, 'share_url', None) or share_url\n    if share_url:\n        break\n    time.sleep(1)\n\nshare_url = share_url or getattr(demo, 'share_url', None)\nprint('READY:')\nprint('Gradio Share URL:', share_url)\nPath('/tmp/bonsai_gradio_url.json').write_text(json.dumps({'share_url': str(share_url) if share_url else None}))\n''')\n\n# 5) install gradio into venv\nrun(f'{VENV_PY} -m pip install -q gradio gradio_client pillow')\n\n# 6) run launcher in background and wait for URL file\nprint('\n--- Launching Gradio ---')\nproc = subprocess.Popen(\n    [str(VENV_PY), str(LAUNCHER)],\n    cwd=str(REPO),\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n)\n\nfor _ in range(180):\n    if URL_FILE.exists():\n        break\n    time.sleep(1)\n\nshare_url = None\nif URL_FILE.exists():\n    try:\n        share_url = json.loads(URL_FILE.read_text()).get('share_url')\n    except Exception as e:\n        print('URL file read failed:', e)\n\nprint('\n' + '='*60)\nprint('READY')\nprint(f'Gradio Share URL: {share_url}')\nprint('='*60)\n